In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt


In [4]:
# Load CSV survey file
df = pd.read_csv(r"C:\Users\Ahmed Hatem\Downloads\diabetes_012_health_indicators_BRFSS2015.csv\diabetes_012_health_indicators_BRFSS2015.csv")

print(df.head())
print(df.columns)
print(df['Diabetes_012'].value_counts()) 

   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3           0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4           0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15.0       1.0  0.0   9.0        4.0   
1     

In [5]:
df

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,2.0,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [6]:
print(df.columns)


Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')


In [7]:
# 0 = no diabetes, 1 = prediabetes or diabetes
df['label'] = df['Diabetes_012'].apply(lambda x: 0 if x == 0 else 1)
y = df['label']


In [8]:
# Insert this new cell after Cell 5 (where you create 'label' and set y = df['label'])

# Compute Pearson correlations with the label
correlations = df.corr(method='pearson')['label'].sort_values(key=abs, ascending=False)
print(correlations)

label                   1.000000
Diabetes_012            0.983304
GenHlth                 0.300785
HighBP                  0.270334
BMI                     0.223851
DiffWalk                0.222155
HighChol                0.210290
Age                     0.185891
HeartDiseaseorAttack    0.176933
PhysHlth                0.174948
Income                 -0.172794
Education              -0.131803
PhysActivity           -0.121392
Stroke                  0.104800
MentHlth                0.074971
CholCheck               0.067879
Smoker                  0.062778
Veggies                -0.059219
HvyAlcoholConsump      -0.056682
Fruits                 -0.042088
NoDocbcCost             0.038025
Sex                     0.029606
AnyHealthcare           0.014079
Name: label, dtype: float64


In [13]:
# Explicitly exclude the original diabetes column and the derived label
exclude = ['Diabetes_012', 'label']

# Manually define your selected top features (from feature importance)
selected_features = [
    'GenHlth',
    'BMI',
    'HighBP',
    'Age',
    'HighChol',
   
    'PhysHlth',
    
    'DiffWalk',
    'PhysActivity',
    'HeartDiseaseorAttack',
    'Stroke'
]

# Remove excluded columns just in case
safe_features = [f for f in selected_features if f not in exclude]

# OPTIONAL: Remove potential post-diagnosis features (safer medically)
safe_features = [f for f in safe_features if f not in ['HeartDiseaseorAttack', 'Stroke']]

X = df[safe_features]

print("Final features used:", safe_features)


Final features used: ['GenHlth', 'BMI', 'HighBP', 'Age', 'HighChol', 'PhysHlth', 'DiffWalk', 'PhysActivity']


In [14]:
# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [29]:
# Combine train features and labels temporarily
train_df = X_train.copy()
train_df['label'] = y_train.values

# Separate classes
df_majority = train_df[train_df['label'] == 0]
df_minority = train_df[train_df['label'] == 1]

print("Before balancing:")
print(df_majority.shape, df_minority.shape)

# Undersample majority
df_majority_downsampled = df_majority.sample(
    n=len(df_minority),
    random_state=42
)

# Combine
balanced_train_df = pd.concat([df_majority_downsampled, df_minority])

# Shuffle
balanced_train_df = balanced_train_df.sample(frac=1, random_state=42)

# Separate again
X_train_balanced = balanced_train_df.drop(columns=['label'])
y_train_balanced = balanced_train_df['label']

print("After balancing:")
print(y_train_balanced.value_counts())


Before balancing:
(170962, 9) (31982, 9)
After balancing:
label
1    31982
0    31982
Name: count, dtype: int64


In [41]:
# Combine test features and labels temporarily
test_df = X_test.copy()
test_df['label'] = y_test.values

# Separate classes
df_test_majority = test_df[test_df['label'] == 0]
df_test_minority = test_df[test_df['label'] == 1]

print("Before balancing test set:")
print(df_test_majority.shape, df_test_minority.shape)

# Undersample majority to match minority
df_test_majority_down = df_test_majority.sample(
    n=len(df_test_minority),
    random_state=42
)

# Combine to create balanced test set
balanced_test_df = pd.concat([df_test_majority_down, df_test_minority])

# Shuffle
balanced_test_df = balanced_test_df.sample(frac=1, random_state=42)

# Separate features and labels
X_test_balanced = balanced_test_df.drop(columns=['label'])
y_test_balanced = balanced_test_df['label']

print("After balancing test set:")
print(y_test_balanced.value_counts())


Before balancing test set:
(42741, 9) (7995, 9)
After balancing test set:
label
1    7995
0    7995
Name: count, dtype: int64


In [44]:
print("Training distribution AFTER balancing:")
print(y_train_balanced.value_counts())

print("\nTest distribution (should remain imbalanced):")
print(y_test_balanced.value_counts())


Training distribution AFTER balancing:
label
1    31982
0    31982
Name: count, dtype: int64

Test distribution (should remain imbalanced):
label
1    7995
0    7995
Name: count, dtype: int64


In [45]:
from sklearn.ensemble import GradientBoostingClassifier

# Fresh model
model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.7,
    random_state=42
)

# Train ONLY on balanced training data
model.fit(X_train_balanced, y_train_balanced)


,loss,'log_loss'
,learning_rate,0.05
,n_estimators,300
,subsample,0.7
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [ ]:
# Probabilities
y_probs = model.predict_proba(X_test_balanced)[:, 1]

# Default threshold 0.5
threshold = 0.5
y_pred = (y_probs >= threshold).astype(int)


In [50]:
print("Balanced train label counts:")
print(y_train_balanced.value_counts())


Balanced train label counts:
label
1    31982
0    31982
Name: count, dtype: int64


In [ ]:
# Predict probabilities on balanced test set
y_probs_bal = model.predict_proba(X_test_balanced)[:, 1]

# Threshold 0.4
y_pred_bal = (y_probs_bal >= 0.4).astype(int)

print("=== Evaluation on Balanced Test Set ===")
print("Accuracy:", accuracy_score(y_test_balanced, y_pred_bal))
print("ROC-AUC:", roc_auc_score(y_test_balanced, y_probs_bal))
print("\nClassification Report:\n", classification_report(y_test_balanced, y_pred_bal))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_balanced, y_pred_bal))


=== Evaluation on Balanced Test Set ===
Accuracy: 0.7328330206378987
ROC-AUC: 0.8172529970068065

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.61      0.70      7995
           1       0.69      0.86      0.76      7995

    accuracy                           0.73     15990
   macro avg       0.75      0.73      0.73     15990
weighted avg       0.75      0.73      0.73     15990


Confusion Matrix:
 [[4873 3122]
 [1150 6845]]
